# 00 认识八界机器人与 SDK

## 学习目标

- 知道 `bajie_sdk` 是什么。
- 知道 `BajieRobot.eco_*` 是主要入口。
- 分清 WebSocket mission 类能力和 HTTP 感知类能力。
- 建立第一组机器人使用习惯。

参考文献：《八界机器人 SDK 开发文档（Python）v0.3.0》第 3、4、6、7 章。

## 本章完成标准

你不需要连接机器人。读完后，你应该能回答：

- SDK 连接的是哪个 IP？
- 哪些接口属于会让机器人执行任务的 mission？
- 为什么不能把一串动作命令连续快速发出去？

## 背景知识

八界机器人 SDK 把机器人能力封装成 Python 方法。多数业务代码只需要：

1. `from bajie_sdk import BajieRobot`
2. 创建 `robot = BajieRobot(ws_url=...)`
3. `robot.Connect()`
4. 调用 `robot.eco_*`
5. `robot.Disconnect()`

机器人能力大致分两类：

- **Mission 类能力**：导航、拍照、抓取、放置、本体控制等。它们通常会产生任务，有 `task_id`，需要等待 `finish`。
- **感知类能力**：开放词检测、VLM 感知、embedding、相似度等。它们通常基于图像走 HTTP 服务。

一个常见闭环是：

`拍照 eco_captureImages` → `检测 eco_detect_objects` → `对准 eco_lookto` → `抓取 eco_pick` → `放置 eco_place_in / eco_place_3D`

## 机器人使用习惯

- 不要连续狂发动作命令。导航、抓取、放置都需要时间完成。
- 做任务前先看状态：电量、告警、是否正在工作、地图/家具是否存在。
- 拍照和检测依赖视角。看不到、被遮挡、光照差，检测就可能失败。
- Linux 板 IP 才是 SDK 连接地址，常见默认值是 `ws://10.10.10.12:9900/`。这个 IP 需要到机器人网页配置页面查询或设置，不要直接使用屏幕显示的安卓 IP。
- 真实机器人旁边有人时再执行移动、抓取、放置；notebook 默认把这些开关关掉。

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from helpers import DEFAULT_WS_URL

WS_URL = DEFAULT_WS_URL
print("WS_URL =", WS_URL)

In [ ]:
# 先不用连接机器人，只确认本教程的默认连接地址。
# 后续每个 notebook 顶部都可以改 WS_URL。
print("SDK 默认机器人地址:", WS_URL)
print("如果不是这个地址，请到机器人网页配置页面查询 Linux 板 IP，然后改 WS_URL。")

## 故障排查卡片

- 不知道接口怎么用：先查 `python -m bajie_sdk -m eco_xxx`。
- 不知道请求字段：再查 `python -m bajie_sdk -t TypeName`。
- 任务失败：先看 `MissionStatus.error_info`。
- 连接失败：先到机器人网页配置页面确认 Linux 板 IP，再检查端口 9900、网络连通性和机器人服务。

## 小练习

用自己的话解释：为什么抓取前通常要先拍照和检测？